## Evaluate fragment linkings

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from plotly import express as px
from tqdm import tqdm

In [ ]:
from tools import compute_uniqueness, compute_novelty, compute_unique_novelty

## Load data

In [ ]:
# load individually evaluation files
methods = ["link_replacment", "link_replacment_no_h"]
dfs = []
for method in methods:
    df = pd.read_csv(f"predictions/conditional_fragments/attempt_2/{method}.csv")
    df = df[~df.fail.fillna(0).astype(bool)].reset_index() # Evaluation script does not break molecules correctly, introducting these rows
    df = df.drop(columns=["index", "fail"])

    df_frag = pd.read_csv(f'predictions/conditional_fragments/attempt_2/{method}_combined.csv', low_memory=False).reset_index()
    df_frag.columns = [c.lower().replace(" ", "_") for c in df_frag.columns]
    df_frag = df_frag.drop(columns=["index", "fail", "error"])

    assert len(df) == len(df_frag), f"Lengths are not equal: {len(df)} != {len(df_frag)}"
    df = pd.concat([df, df_frag], axis=1, ignore_index=False)

    df["method"] = method
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

df["total"] = True
df["valid"] = df["connected"] & df["chemical"] & df["physical"]

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df["num_atoms_missing"] = df["num_atoms_link"] - df["num_atoms_frag"]
df["share_atoms_frag"] = df["num_atoms_frag"] / df["num_atoms_link"]
df["share_atom_missing"] = df["num_atoms_missing"] / df["num_atoms_link"]

# Table: validity

In [ ]:
df_valid = df[['method',"total", 'connected', 'chemical', 'physical', "valid"]].groupby("method").sum()
df_valid.index.name = None
print(df_valid)

print(" --- --- --- --- --- ")
df_valid = df[['method',"total", 'connected', 'chemical', 'physical', "valid"]].groupby("method").mean()
df_valid.index.name = None
print(df_valid)

# Table: scaffold hop

In [ ]:
df_scaff = df[df.valid][["method", "scaffold_true_csk", "scaffold_rdkit_csk", "sucos_frag", "sucos_link"]].groupby("method").mean()
df_scaff.style.format("{:.2%}")

# Compute plots

In [ ]:
sns.histplot(df, x="tanimoto_frag", hue="method", element="step", common_norm=False, bins=100)

In [ ]:
sns.histplot(df, x="tanimoto_link", hue="method", element="step", common_norm=False, bins=100)

In [ ]:
sns.histplot(df, x="sucos_frag", hue="method", element="step", common_norm=False, bins=100)

In [ ]:
sns.histplot(df, x="sucos_link", hue="method", element="step", common_norm=False, bins=100)

In [ ]:
# How much repetition is there? How unique are the generated molecules?
s = df.groupby("method")["smiles_pred"].agg(compute_uniqueness)
s.name = "Uniqueness"
s

In [ ]:
# How many of the valid generated molecules are not in the test set?
s = df.groupby("method")["smiles_pred"].agg(compute_novelty)
s.name = "Novelty"
s

In [ ]:
#

In [ ]:
#

## Metrics

In [ ]:
# metrics = {"sucos": "SuCOS", "tanimoto": "ECFP4 Bit Tanimoto"}

In [ ]:
# metric = "sucos"
# name = metrics[metric]

# g = sns.FacetGrid(df, col="comparison", hue="method", height=5, aspect=1.3)
# g.map(
#     sns.histplot,
#     metric,
#     bins=50,
#     common_norm=False,
#     stat="density",
#     element="step",
#     # kde=True,
#     fill=False,
# )
# g.add_legend()

In [ ]:
# metric = "tanimoto"
# name = metrics[metric]

# g = sns.FacetGrid(df, col="comparison", hue="method", height=5, aspect=1.3)
# g.map(
#     sns.histplot,
#     metric,
#     bins=50,
#     common_norm=False,
#     stat="density",
#     element="step",
#     # kde=True,
#     fill=False,
# )
# g.add_legend()

In [ ]:
# g = sns.FacetGrid(df, col="method", row="comparison", height=3, aspect=1.3)
# g.map(sns.kdeplot, "num_atoms_cond", "num_atoms_pred")

In [ ]:
#

In [ ]:
#

In [ ]:
#

In [ ]:
# ## Try pairing the data
# pred_dir = Path("/homes/buttensc/Projects/semla-flow/predictions/fragment")
# files = list(pred_dir.glob("*.csv"))
# dfs_linker, dfs_fragment = [], []
# for file in tqdm(files):
#     comparison = parts[-1]
#     if comparison == "linker":
#         df = pd.read_csv(file)
#         parts = Path(file).stem.split("_")
#         df["method"] = " ".join(parts[0:2]) + " " + (parts[3] if len(parts) > 4 else "")
#         df = df.dropna(subset=["Reference molecule"])
#         dfs_linker.append(df)
#     elif comparison == "fragment":
#         df = pd.read_csv(file)
#         parts = Path(file).stem.split("_")
#         df["method"] = " ".join(parts[0:2]) + " " + (parts[3] if len(parts) > 4 else "")
#         df = df.dropna(subset=["Reference molecule"])
#         dfs_fragment.append(df)
# df_linker = (
#     pd.concat(dfs_linker)
#     .sort_values(["method", "Reference molecule"])
#     .set_index(["smiles_pred", "Reference molecule"])
# )
# df_fragment = (
#     pd.concat(dfs_fragment)
#     .sort_values(["method", "Reference molecule"])
#     .set_index(["smiles_pred", "Reference molecule"])
# )


In [ ]:
#